<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>99101 עיבוד שפה טבעית NLP</p>
<p>פרויקט סיום</p>
<p>שם: נדב פיירמן שטרן</p>
<hr>
</div>

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>מחברת זו אוספת את נתוני הגלם של הפרויקט מתוך ויקי One Piece, ומיועדת לריצה אחת בלבד. הפלט הוא קובץ אחד הכולל לכל דמות את השם, מזהה הדף, הכתובת והטקסט הגולמי המלא.</p>
<p>מחברת זו הינה המחברת המשנית. קיימת מחברת נוספת העוסקת בכל החיתוך ועיבוד של המידע, שהינה המחברת הראשית.</p>
<ul>
<li>שלב א — איסוף רשימת כל דפי הדמויות: שם, מזהה וכתובת</li>
<li>שלב ב — משיכת הטקסט הגולמי המלא של כל דף, באצוות של חמישים דפים לכל בקשה</li>
<li>שחזור תיבות מידע חוץ-עמודיות עבור דפי הדמויות הגדולות, שתיבתם מוטמעת מתבנית נפרדת</li>
<li>שחזור גוף הטקסט (אישיות ויחסים, היסטוריה, יכולות) עבור דפי הדמויות הגדולות, ששמור בתת-דפים נפרדים</li>
</ul>
<hr>
</div>

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>התקנת הספרייה החיצונית הדרושה לאיסוף, שאינה מותקנת כברירת מחדל בסביבת Colab.</p>
</div>

In [ ]:
!pip install -q mwparserfromhell

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>ייבוא הספריות הדרושות למחברת, שהן גם הספריות שבהן משתמש קובץ הקוד data_collection:</p>
<ul>
<li>os, sys, re, time — ספריות התקן לעבודה עם נתיבים, טקסט וזמן</li>
<li>requests — שליחת בקשות HTTP אל ממשק הוויקי</li>
<li>pandas — ארגון הנתונים בטבלה ושמירתם כקובץ CSV</li>
</ul>
</div>

In [ ]:
import os, sys, re, time
import requests
import pandas as pd

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>חיבור ל-Google Drive לצורך טעינת קובץ הקוד data_collection ושמירת קובץ הפלט.</p>
</div>

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>הגדרת נתיב הפרויקט והוספתו לנתיב החיפוש של פייתון, כך שניתן לייבא ממנו את קובץ הקוד. כל הקבצים, כולל קובץ הפלט, נשמרים יחד בתיקיית הפרויקט. בסיום מודפסים הנתיב ותוכן התיקייה לאימות.</p>
</div>

In [ ]:
PATH = '/content/drive/MyDrive/NLP/Project'

if PATH not in sys.path:
    sys.path.insert(0, PATH)

print('Files in project:', os.listdir(PATH))

Files in project: ['data_collection.ipynb', 'data_collection.py', 'onepiece_raw.csv', 'modeling.py', 'network_plots.py', 'REFLECTION.md', 'DATA.md', 'ETHICS.md', 'AI_USAGE.md', 'text_features.py', 'features.py', 'one_piece_nlp_project.ipynb']


<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>ייבוא פונקציות האיסוף מתוך קובץ הקוד data_collection. שימוש ב-reload מבטיח שעריכות בקובץ ה-PY ייטענו מחדש ללא צורך באתחול הסביבה. הפונקציות:</p>
<ul>
<li>fetch_all_pages — איסוף רשימת כל דפי הדמויות (שלב א)</li>
<li>fetch_descriptions — משיכת הטקסט הגולמי לכל דף (שלב ב)</li>
<li>fetch_tabs_infoboxes — שחזור תיבת המידע החוץ-עמודית עבור דפי הדמויות הגדולות</li>
<li>fetch_subpages — שחזור גוף הטקסט מתת-הדפים עבור דפי הדמויות הגדולות</li>
<li>drop_noise — הסרת דפי רעש (גלריות, תת-דפים והפניות ריקות) כדי לשמור קובץ נקי</li>
</ul>
</div>

In [ ]:
import importlib
import data_collection
importlib.reload(data_collection)

from data_collection import fetch_all_pages, fetch_descriptions, fetch_tabs_infoboxes, fetch_subpages, drop_noise

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>הגדרת הקטגוריות לאיסוף ונתיב קובץ הפלט. קטגוריית העל Characters מכילה תת-קטגוריות בלבד וללא דפים ישירים, ולכן האיסוף יורד רקורסיבית אל תת-הקטגוריות שתחתיה (כגון Male Characters ו-Female Characters), המכסות יחד את כלל הדמויות בוויקי. ניתן להוסיף קטגוריות לרשימה ללא שינוי שאר הקוד.</p>
</div>

In [ ]:
CATEGORIES = ["Characters"]

RAW_CSV = PATH + "/onepiece_raw.csv"

print("Categories:", CATEGORIES)

Categories: ['Characters']


<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>שלב א — איסוף רשימת הדפים. עבור כל קטגוריה נאספים כל הדפים שבה, תוך הסרת כפילויות. התוצאה היא רשימה ובה שם הדמות, מזהה הדף וכתובתו. שלב זה מהיר ודורש מספר בקשות בלבד.</p>
</div>

In [ ]:
pages = fetch_all_pages(CATEGORIES)
print("Collected", len(pages), "pages")

  Characters                             0 pages,     0 new
    Antagonists                            1 pages,     1 new
      Non-Canon Antagonists                 46 pages,    46 new
      Antagonist Groups                     66 pages,    66 new
      Antagonists by Saga                    0 pages,     0 new
      Cover Story Antagonists               29 pages,    29 new
      Flashback Antagonists                 36 pages,    33 new
      One-Shot Antagonists                  12 pages,    10 new
    Character Galleries                   14 pages,    14 new
    Characters by Ability                  0 pages,     0 new
      Non-Canon Characters by Ability        0 pages,     0 new
      Devil Fruit Users                      3 pages,     2 new
      Fighters Who Use Animals              24 pages,    22 new
      Fighters Who Use Technology           17 pages,    17 new
      Fighters Who Use Weapons               8 pages,     8 new
      Haki Users                             8 pag

<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>שלב ב — משיכת הטקסט הגולמי המלא של כל דף. השלב כולל את הפעולות הבאות:</p>
<ul>
<li>משיכת הטקסט הגולמי המלא של כל דף, באצוות של חמישים דפים לכל בקשה, כך שאלפי דפים נמשכים בעשרות בקשות בלבד.</li>
<li>עבור דפי הדמויות הגדולות שתיבת המידע שלהם מוטמעת מתוך עמוד תבנית נפרד — נמשכת גם התבנית ומשורשרת לטקסט, כך שתיבת המידע המלאה (היסטוריית הבאונטי, התפקיד והשיוך) זמינה לעיבוד.</li>
<li>עבור אותן דמויות גדולות, גם גוף הטקסט (אישיות ויחסים, היסטוריה ויכולות) אינו נמצא בדף הראשי אלא בתת-דפים נפרדים — תת-דפים אלה נאספים ומשורשרים לטקסט, כך ש-description מכיל את הדף המלא ולא רק את תיבת המידע.</li>
<li>הסרת דפי רעש — דפי גלריה ותת-דפים ששמם מכיל לוכסן, ודפי הפניה ריקים — כך שנשמרים רק דפי דמויות וצוותים בעלי תוכן.</li>
<li>שמירת התוצאה לקובץ הכולל את העמודות name, pageid, url, description.</li>
</ul>
</div>

In [ ]:
pages = fetch_descriptions(pages)
pages = fetch_tabs_infoboxes(pages)
pages = fetch_subpages(pages)
pages = drop_noise(pages)

raw_df = pd.DataFrame(pages)[["name", "pageid", "url", "description"]]
raw_df.to_csv(RAW_CSV, index=False, encoding="utf-8-sig")

print("Saved", len(raw_df), "characters to", RAW_CSV)
print("Empty description:", (raw_df['description'].fillna('').str.len() == 0).sum())
raw_df.head()

  fetched    50 / 1770
  fetched   100 / 1770
  fetched   150 / 1770
  fetched   200 / 1770
  fetched   250 / 1770
  fetched   300 / 1770
  fetched   350 / 1770
  fetched   400 / 1770
  fetched   450 / 1770
  fetched   500 / 1770
  fetched   550 / 1770
  fetched   600 / 1770
  fetched   650 / 1770
  fetched   700 / 1770
  fetched   750 / 1770
  fetched   800 / 1770
  fetched   850 / 1770
  fetched   900 / 1770
  fetched   950 / 1770
  fetched  1000 / 1770
  fetched  1050 / 1770
  fetched  1100 / 1770
  fetched  1150 / 1770
  fetched  1200 / 1770
  fetched  1250 / 1770
  fetched  1300 / 1770
  fetched  1350 / 1770
  fetched  1400 / 1770
  fetched  1450 / 1770
  fetched  1500 / 1770
  fetched  1550 / 1770
  fetched  1600 / 1770
  fetched  1650 / 1770
  fetched  1700 / 1770
  fetched  1750 / 1770
  fetched  1770 / 1770
Tabs pages: 38, unique templates: 28
  fetched    28 / 28 templates
Tabs (major) characters needing subpages: 38
  content subpages found: 100
  fetched    50 / 100 subpage

,name,pageid,url,description
0,Draw,302300,https://onepiece.fandom.com/wiki/Draw,{{Char Box\n|colorscheme = MarinesColors\n|swi...
1,150,325159,https://onepiece.fandom.com/wiki/150,{{Char Box\n|colorscheme = OrganDealingAs...
2,All-Hunt Grount,295140,https://onepiece.fandom.com/wiki/All-Hunt_Grount,{{Char Box\n| colorscheme = MarinesColors\n| j...
3,Ant De Bonham,295141,https://onepiece.fandom.com/wiki/Ant_De_Bonham,{{Char Box\n| colorscheme = MarinesColors\n| j...
4,Artur Bacca,318276,https://onepiece.fandom.com/wiki/Artur_Bacca,{{Stub|Character}}\n{{Char Box\n| colorscheme ...


<div dir="rtl" align="right" style="font-family: Calibri, sans-serif; font-size: 12pt; line-height: 1.5;">
<p>בדיקת שפיות — אימות שהאיסוף הצליח. נבדק מספר הדמויות, מספר הדמויות בעלות תוכן ואורך הטקסט הממוצע, מאומת שדמויות מפתח נאספו, ונבדק ששחזור התיבות הצליח עבור דמות בעלת דף Tabs. כמו כן נבדק שדפי הדמויות הגדולות קיבלו את גוף הטקסט המלא ולא רק תיבת מידע, על ידי איתור דפי Tabs שטקסטם עדיין קצר.</p>
</div>

In [ ]:
print("Total characters :", len(raw_df))
print("With description :", (raw_df['description'].fillna('').str.len() > 0).sum())
print("Avg length       :", int(raw_df['description'].fillna('').str.len().mean()), "chars")

for nm in ["Monkey D. Luffy", "Kaidou", "Shanks", "Crocodile", "Jinbe", "Buggy"]:
    n = raw_df['name'].str.contains(nm, case=False, na=False).sum()
    print(f"{nm:18}: {n} match(es)")

buggy = raw_df[raw_df['name'] == 'Buggy']
if len(buggy):
    print("\nBuggy Char Box recovered:", "Char Box" in buggy.iloc[0]['description'])

# big (Tabs) characters must now carry full body text, not just an infobox
desc = raw_df['description'].fillna('')
tabbed = desc.str.contains(r"\{\{[^{}]*Tabs Top\}\}", regex=True)
thin = raw_df[tabbed & (desc.str.len() < 1000)]
print("\nTabs characters under 1000 chars:", len(thin))
if len(thin):
    print(thin['name'].head(20).tolist())

Total characters : 1734
With description : 1734
Avg length       : 15420 chars
Monkey D. Luffy   : 1 match(es)
Kaidou            : 1 match(es)
Shanks            : 1 match(es)
Crocodile         : 1 match(es)
Jinbe             : 1 match(es)
Buggy             : 3 match(es)

Buggy Char Box recovered: True

Tabs characters under 1000 chars: 0
